# AI-Powered Resume Evaluation & ATS Optimization System

## Project Objective

The objective of this project is to develop an intelligent Resume Evaluation System that automates the recruitment screening process using Machine Learning, Natural Language Processing (NLP), Sentence-BERT, and Google's Gemini Large Language Model (LLM).

The system classifies resumes into different job roles, compares resumes with a given Job Description (JD), performs semantic similarity analysis, identifies missing technical skills, calculates an ATS compatibility score, and generates AI-powered feedback to help candidates improve their resumes.

---

## Key Features

- Resume Classification using Machine Learning
- Resume PDF Upload and Text Extraction
- Job Description Processing
- Sentence-BERT Semantic Similarity
- Dynamic Skill Extraction
- Semantic Skill Matching
- ATS Score Calculation
- AI-Powered Resume Feedback using Gemini
- AI Resume Chatbot

---

## Technologies Used

### Programming Language
- Python

### Machine Learning
- Scikit-learn
- Logistic Regression
- TF-IDF Vectorizer

### Natural Language Processing
- Sentence Transformers (Sentence-BERT)

### Large Language Model
- Google Gemini 2.5 Flash

### Libraries
- Pandas
- NumPy
- Joblib
- PyMuPDF (fitz)
- Scikit-learn
- Sentence-Transformers
- Google Generative AI

---

## Project Workflow

Dataset
↓
Data Preprocessing
↓
Resume Classification Model
↓
Resume Upload
↓
Job Description Input
↓
Sentence-BERT Semantic Matching
↓
Dynamic Skill Matching
↓
ATS Score Calculation
↓
RAG Context Generation
↓
Gemini AI Feedback
↓
AI Resume Chatbot

---

## Project Modules

1. Installation & Imports
2. Dataset Loading
3. Data Preprocessing
4. Resume Classification Model
5. Save & Load Models
6. Resume Upload & PDF Extraction
7. Job Description Processing
8. Sentence-BERT Semantic Matching
9. Dynamic Skill Matching
10. ATS Score Calculation
11. Gemini Configuration
12. Core AI Engine
13. Complete Resume Evaluation
14. AI Resume Chatbot

# Section 2 – Installation & Imports

In [1]:
!pip install sentence-transformers
!pip install google-generativeai
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 47.2 MB/s eta 0:00:00


In [2]:
# ============================
# Standard Libraries
# ============================

import os
import json
import re
import warnings

warnings.filterwarnings("ignore")

# ============================
# Data Handling
# ============================

import numpy as np
import pandas as pd

# ============================
# Machine Learning
# ============================

import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report
)

# ============================
# NLP
# ============================

from sentence_transformers import SentenceTransformer, util

# ============================
# PDF Processing
# ============================

import fitz

# ============================
# Google Gemini
# ============================

import google.generativeai as genai

# ============================
# Utilities
# ============================

from pprint import pprint

# Section 3 – Dataset Loading

In [3]:
from google.colab import files

uploaded = files.upload()

Saving resumes_dataset.jsonl to resumes_dataset.jsonl


In [4]:
import json

resumes = []

with open("resumes_dataset.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        resumes.append(json.loads(line))

df = pd.DataFrame(resumes)

print("Dataset Loaded Successfully!")
print("Total Records:", len(df))

Dataset Loaded Successfully!
Total Records: 3500


In [5]:
print("Dataset Shape:", df.shape)

display(df.head())

df.info()

print("\nMissing Values")
print(df.isnull().sum())

Dataset Shape: (3500, 12)


,ResumeID,Category,Name,Email,Phone,Location,Summary,Skills,Experience,Education,Text,Source
0,REAL_0001,Java Developer,Chad Griffin,contact@email.com,94105 555 4321000 10 ...,"City, State",jessica claire montgomery street san francisco...,"Python, SQL, Git, Linux",jessica claire montgomery street san francisco...,Computer Science degree,jessica claire montgomery street san francisco...,ResumeAtlas
1,REAL_0002,Java Developer,Melinda Thomas,contact@email.com,17994568777 2017 2018 20152016 3 ...,"City, State",jared arthur maica java developer 17994568777 ...,"Python, SQL, Git, Linux",jared arthur maica java developer 17994568777 ...,Computer Science degree,jared arthur maica java developer 17994568777 ...,ResumeAtlas
2,REAL_0003,Java Developer,Shannon Mccarthy,contact@email.com,9 555 4321000 94105 8 ...,"City, State",jessica claire 9 resumesampleexamplecom 555 43...,"Python, SQL, Git, Linux",jessica claire 9 resumesampleexamplecom 555 43...,Computer Science degree,jessica claire 9 resumesampleexamplecom 555 43...,ResumeAtlas
3,REAL_0004,Java Developer,Christine Kelley,contact@email.com,9 555 4321000 94105 5 ...,"City, State",jessica claire 9 resumesampleexamplecom 555 43...,"Python, SQL, Git, Linux",jessica claire 9 resumesampleexamplecom 555 43...,Computer Science degree,jessica claire 9 resumesampleexamplecom 555 43...,ResumeAtlas
4,REAL_0005,Java Developer,Karen Holt,contact@email.com,100 10 4321000 ...,"City, State",jessica claire 100 montgomery st 10th floor xx...,"Python, SQL, Git, Linux",jessica claire 100 montgomery st 10th floor xx...,Computer Science degree,jessica claire 100 montgomery st 10th floor xx...,ResumeAtlas


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3500 entries, 0 to 3499
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   ResumeID    3500 non-null   object
 1   Category    3500 non-null   object
 2   Name        3500 non-null   object
 3   Email       3500 non-null   object
 4   Phone       3500 non-null   object
 5   Location    3500 non-null   object
 6   Summary     3500 non-null   object
 7   Skills      3500 non-null   object
 8   Experience  3500 non-null   object
 9   Education   3500 non-null   object
 10  Text        3500 non-null   object
 11  Source      3500 non-null   object
dtypes: object(12)
memory usage: 328.3+ KB

Missing Values
ResumeID      0
Category      0
Name          0
Email         0
Phone         0
Location      0
Summary       0
Skills        0
Experience    0
Education     0
Text          0
Source        0
dtype: int64


In [6]:
df = df[
    [
        "ResumeID",
        "Category",
        "Skills",
        "Experience",
        "Education",
        "Text"
    ]
]

# Section 4 – Resume Classification Model Preparation

In [7]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"www\S+", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [8]:
df["Cleaned_Text"] = df["Text"].apply(clean_text)

print("Sample Cleaned Resume:\n")
print(df["Cleaned_Text"].iloc[0][:500])

Sample Cleaned Resume:

jessica claire montgomery street san francisco ca resumesampleexamplecom professional summary highly skilled software development professional bringing years software design development integration advanced knowledge java skills agile html xml jdbc tomcat work history senior java developertech lead current synnex corporation tracy ca java developer agile scrum team javascript java develop customer facing internal web applications underlying component applications wrote maintainable extensible co


In [9]:
label_encoder = LabelEncoder()

df["Encoded_Category"] = label_encoder.fit_transform(
    df["Category"]
)

print("Number of Categories:", len(label_encoder.classes_))
print(label_encoder.classes_)

Number of Categories: 36
['AI Engineer' 'Backend Developer' 'Blockchain' 'Blockchain Developer'
 'Business Analyst' 'Cloud Engineer' 'Cybersecurity Analyst'
 'Data Science' 'Database' 'Database Administrator' 'DevOps'
 'Digital Media' 'DotNet Developer' 'ETL Developer' 'Engineering Manager'
 'Frontend Developer' 'Full Stack Developer' 'Java Developer'
 'Machine Learning Engineer' 'Mobile Developer'
 'Network Security Engineer' 'Principal Engineer' 'Product Manager'
 'Python Developer' 'QA Engineer' 'React Developer' 'SAP Developer'
 'SQL Developer' 'Site Reliability Engineer' 'Software Developer'
 'System Administrator' 'Technical Lead' 'Technical Writer' 'Testing'
 'UI/UX Designer' 'Web Designing']


In [10]:
X = df["Cleaned_Text"]

y = df["Encoded_Category"]

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Samples :", len(X_train))
print("Testing Samples  :", len(X_test))

Training Samples : 2800
Testing Samples  : 700


# Section 5 – Resume Classification Model

This section trains a Machine Learning model to classify resumes into their corresponding job categories. A TF-IDF Vectorizer converts resume text into numerical feature vectors, and a Logistic Regression classifier is trained to predict the most suitable job role.

In [12]:
vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    stop_words="english",
    min_df=2
)

X_train_vector = vectorizer.fit_transform(X_train)
X_test_vector = vectorizer.transform(X_test)

In [13]:
model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_train_vector, y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [14]:
y_pred = model.predict(X_test_vector)

In [15]:
accuracy = accuracy_score(y_test, y_pred)

print(f"Model Accuracy : {accuracy*100:.2f}%")

Model Accuracy : 88.57%


In [16]:
print(classification_report(
    y_test,
    y_pred,
    target_names=label_encoder.classes_
))

                           precision    recall  f1-score   support

              AI Engineer       1.00      1.00      1.00        14
        Backend Developer       1.00      1.00      1.00        15
               Blockchain       1.00      0.70      0.82        10
     Blockchain Developer       1.00      1.00      1.00         6
         Business Analyst       0.83      1.00      0.91        30
           Cloud Engineer       1.00      1.00      1.00        19
    Cybersecurity Analyst       1.00      1.00      1.00        13
             Data Science       0.83      0.95      0.88        40
                 Database       0.85      0.57      0.68        30
   Database Administrator       1.00      1.00      1.00        11
                   DevOps       0.97      1.00      0.99        36
            Digital Media       0.87      0.65      0.74        20
         DotNet Developer       0.79      0.82      0.81        28
            ETL Developer       0.75      0.88      0.81     

In [17]:
joblib.dump(model, "resume_classifier.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")
joblib.dump(label_encoder, "label_encoder.pkl")

print("Models saved successfully.")

Models saved successfully.


In [18]:
model = joblib.load("resume_classifier.pkl")
vectorizer = joblib.load("tfidf_vectorizer.pkl")
label_encoder = joblib.load("label_encoder.pkl")

print("Models loaded successfully.")

Models loaded successfully.


# Section 6 – Resume Upload & PDF Extraction

This module allows the user to upload a resume in PDF format. The uploaded resume is processed using PyMuPDF to extract text, which is then used throughout the ATS evaluation pipeline.

In [19]:
from google.colab import files

uploaded = files.upload()

resume_file = list(uploaded.keys())[0]

print("Uploaded Resume:", resume_file)

Saving ronak.profilesum.pdf to ronak.profilesum.pdf
Uploaded Resume: ronak.profilesum.pdf


In [20]:
def extract_resume_text(pdf_path):

    document = fitz.open(pdf_path)

    text = ""

    for page in document:
        text += page.get_text()

    document.close()

    return text

In [21]:
resume_text = extract_resume_text(resume_file)

print("Resume Extracted Successfully!\n")

print(resume_text[:1000])

Resume Extracted Successfully!

 
Technical Skills 
Basic: HTML5, CSS3, JavaScript, Bootstrap, C Programming, Microsoft Office 
(Word, Excel, PowerPoint), Git & GitHub. 
Intermediate: Python, Java, SQL (MySQL, PostgreSQL), React.js, Node.js, 
.NET Framework, Machine Learning, Data Analysis (Pandas, NumPy), Power BI, 
Jupyter Notebook, Linux (Ubuntu). 
Skilled: Full Stack Web Development, REST API Development, AI & Machine 
Learning, Cybersecurity Fundamentals, Database Design & Management, 
Prisma ORM, Supabase, Responsive Web Design, Problem Solving & 
Debugging, Version Control (Git). 
 
Projects 
1. AI Resume Builder 
Developed an AI-powered web application to help users create professional 
resumes with smart content suggestions. Implemented a responsive interface for 
resume customization and analysis. Technologies Used: Node.js, React.js, 
JavaScript, HTML, CSS. 
2. Student Virtual Online Community 
Built an online platform that allows students to connect, collaborate, and share 

In [22]:
resume_text = clean_text(resume_text)

print("Cleaned Resume Preview:\n")
print(resume_text[:1000])

Cleaned Resume Preview:

technical skills basic html css javascript bootstrap c programming microsoft office word excel powerpoint git github intermediate python java sql mysql postgresql react js node js net framework machine learning data analysis pandas numpy power bi jupyter notebook linux ubuntu skilled full stack web development rest api development ai machine learning cybersecurity fundamentals database design management prisma orm supabase responsive web design problem solving debugging version control git projects ai resume builder developed an ai powered web application to help users create professional resumes with smart content suggestions implemented a responsive interface for resume customization and analysis technologies used node js react js javascript html css student virtual online community built an online platform that allows students to connect collaborate and share academic resources implemented features such as user authentication discussion forums and knowledge 

# Section 7 – Job Description Input & Processing

This module accepts the Job Description (JD) provided by the recruiter. The text is cleaned using the same preprocessing function applied to resumes, ensuring fair comparison during semantic matching and skill extraction.

In [23]:
job_description = """
Paste the complete Job Description here.

Example:

We are looking for a Python Developer with experience in Python,
Machine Learning, SQL, Git, Linux, REST APIs, Docker,
FastAPI, AWS, and problem-solving skills.
"""

In [24]:
print("Original Job Description:\n")
print(job_description)

Original Job Description:


Paste the complete Job Description here.

Example:

We are looking for a Python Developer with experience in Python,
Machine Learning, SQL, Git, Linux, REST APIs, Docker,
FastAPI, AWS, and problem-solving skills.



In [25]:
job_description = clean_text(job_description)

print("Cleaned Job Description:\n")
print(job_description)

Cleaned Job Description:

paste the complete job description here example we are looking for a python developer with experience in python machine learning sql git linux rest apis docker fastapi aws and problem solving skills


In [26]:
print("Characters :", len(job_description))
print("Words :", len(job_description.split()))

Characters : 198
Words : 32


# Section 8 – Sentence-BERT Semantic Matching

This module measures the semantic similarity between the uploaded resume and the job description using the Sentence-BERT (SBERT) model. Unlike keyword matching, Sentence-BERT captures the contextual meaning of sentences, allowing the system to compare resumes and job descriptions more intelligently.

In [27]:
semantic_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Sentence-BERT model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Sentence-BERT model loaded successfully.


In [28]:
resume_embedding = semantic_model.encode(
    resume_text,
    convert_to_tensor=True
)

jd_embedding = semantic_model.encode(
    job_description,
    convert_to_tensor=True
)

In [29]:
semantic_score = util.cos_sim(
    resume_embedding,
    jd_embedding
).item()

semantic_percentage = round(semantic_score * 100, 2)

print("Semantic Match :", semantic_percentage, "%")

Semantic Match : 41.4 %


In [30]:
if semantic_percentage >= 80:
    print("Excellent Semantic Match")
elif semantic_percentage >= 60:
    print("Good Semantic Match")
elif semantic_percentage >= 40:
    print("Moderate Semantic Match")
else:
    print("Low Semantic Match")

Moderate Semantic Match


# Section 9 – Dynamic Skill Matching

This module extracts technical skills from both the resume and the job description using a master skill database and skill alias mapping. It then compares the extracted skills to calculate the skill match percentage, identify matched skills, and highlight missing skills required by the recruiter.

In [31]:
import pandas as pd

master_skills = set()

for skill_text in df["Skills"].dropna():

    if isinstance(skill_text, str):

        skills = skill_text.split(",")

        for skill in skills:

            skill = skill.strip().lower()

            if len(skill) > 1:
                master_skills.add(skill)

master_skills = sorted(master_skills)

print("Total Skills:", len(master_skills))
print(master_skills[:50])

Total Skills: 95
['adobe xd', 'agile', 'airflow', 'android studio', 'angular', 'api testing', 'automation testing', 'aws', 'azure', 'backup recovery', 'cloudformation', 'cryptography', 'css', 'data migration', 'database design', 'django', 'docker', 'ec2', 'ethereum', 'express', 'figma', 'firebase', 'flask', 'flutter', 'ganache', 'gcp', 'git', 'go', 'grafana', 'html', 'html/css', 'iam', 'illustrator', 'incident management', 'invision', 'java', 'javascript', 'jira', 'junit', 'kali linux', 'keras', 'kotlin', 'kubernetes', 'lambda', 'linux', 'manual testing', 'metasploit', 'microservices', 'mlflow', 'mongodb']


In [32]:
import joblib

joblib.dump(master_skills, "master_skill_database.pkl")

print("Skill database saved successfully!")

Skill database saved successfully!


In [33]:
master_skills = joblib.load("master_skill_database.pkl")

print(len(master_skills))

95


In [34]:
for i in range(94):
    print(master_skills[i])

adobe xd
agile
airflow
android studio
angular
api testing
automation testing
aws
azure
backup recovery
cloudformation
cryptography
css
data migration
database design
django
docker
ec2
ethereum
express
figma
firebase
flask
flutter
ganache
gcp
git
go
grafana
html
html/css
iam
illustrator
incident management
invision
java
javascript
jira
junit
kali linux
keras
kotlin
kubernetes
lambda
linux
manual testing
metasploit
microservices
mlflow
mongodb
mysql
nessus
network security
node.js
oracle
penetration testing
performance tuning
photoshop
postgresql
postman
prometheus
prototyping
python
pytorch
query optimization
react
react native
redis
rest api
risk assessment
s3
sass
scrum
selenium
siem
sketch
smart contracts
solidity
splunk
sql
sql server
swift
tensorflow
terraform
testng
truffle
typescript
user research
vpc
vue.js
web3.js
webpack
wireframing
wireshark


In [35]:
additional_skills = [

# Programming Languages
"python","java","c","c++","c#","javascript","typescript","go","golang",
"rust","php","ruby","kotlin","swift","scala","r","matlab","perl","bash",

# Frontend
"html","css","bootstrap","tailwind css","sass","react","react.js","next.js",
"vue","vue.js","angular","jquery","redux","material ui","chakra ui",

# Backend
"node.js","express.js","django","flask","fastapi","spring","spring boot",
"laravel","asp.net","asp.net core","nestjs","graphql","rest api","grpc",

# Databases
"mysql","postgresql","mongodb","sqlite","oracle","sql server","redis",
"firebase","supabase","cassandra","dynamodb","neo4j","elasticsearch",

# Cloud
"aws","amazon web services","azure","gcp","google cloud","cloud computing",
"ec2","s3","lambda","cloudformation",

# DevOps
"docker","kubernetes","jenkins","github actions","gitlab ci","terraform",
"ansible","prometheus","grafana","nginx","apache","ci/cd",

# Version Control
"git","github","gitlab","bitbucket",

# AI / ML
"machine learning","deep learning","artificial intelligence","computer vision",
"nlp","generative ai","transformers","tensorflow","keras","pytorch",
"scikit-learn","xgboost","lightgbm","catboost","opencv","hugging face",

# LLM Ecosystem
"langchain","llamaindex","crewai","autogen","langgraph","ollama",
"openai api","gemini api","claude api","mistral","groq","pinecone",
"chromadb","faiss","vector database","embedding","rag","prompt engineering",

# Data Science
"numpy","pandas","matplotlib","seaborn","plotly","power bi","tableau",
"excel","spark","hadoop","airflow","databricks",

# Cybersecurity
"cybersecurity","network security","ethical hacking","penetration testing",
"owasp","burp suite","wireshark","nmap","metasploit","kali linux",
"linux","windows server","active directory","firewall","vpn",

# Mobile
"android","flutter","react native","ios","xamarin",

# Testing
"selenium","playwright","pytest","junit","postman","insomnia",

# Misc
"microservices","design patterns","oauth","jwt","socket.io",
"rabbitmq","kafka","prisma","streamlit","fastapi","grpc"
]

In [36]:
master_skills = sorted(
    set(master_skills) |
    set([skill.lower() for skill in additional_skills])
)

print("Total Skills:", len(master_skills))

Total Skills: 210


In [37]:
skill_alias = {

    "nodejs":"node.js",
    "node js":"node.js",

    "reactjs":"react",
    "react.js":"react",

    "vuejs":"vue",
    "vue.js":"vue",

    "ml":"machine learning",

    "ai":"artificial intelligence",

    "tf":"tensorflow",

    "sklearn":"scikit-learn",

    "postgres":"postgresql",

    "mongo":"mongodb",

    "gcp":"google cloud",

    "aws":"amazon web services",

    "rest":"rest api",

    "js":"javascript"
}

In [38]:
normalized_skills = []

for skill in master_skills:

    skill = skill.lower().strip()

    if skill in skill_alias:
        skill = skill_alias[skill]

    normalized_skills.append(skill)

master_skills = sorted(set(normalized_skills))

print("Final Skills:", len(master_skills))

Final Skills: 206


In [39]:
import joblib

joblib.dump(master_skills, "master_skill_database.pkl")
joblib.dump(skill_alias, "skill_alias.pkl")

print("Knowledge Base Saved Successfully!")

Knowledge Base Saved Successfully!


In [42]:
master_skills = joblib.load("master_skill_database.pkl")
skill_alias = joblib.load("skill_alias.pkl")

print("Knowledge Base Loaded Successfully!")

Knowledge Base Loaded Successfully!


In [49]:
def extract_resume_skills(text):

    text = text.lower()

    extracted = set()

    for skill in master_skills:

        pattern = r"\b" + re.escape(skill) + r"\b"

        if re.search(pattern, text):
            extracted.add(skill)

    return sorted(extracted)

In [50]:
resume_skills = extract_resume_skills(resume_text)

print("Resume Skills")
print("="*40)

for skill in resume_skills:
    print("•", skill)

Resume Skills
• bootstrap
• c
• css
• cybersecurity
• database design
• excel
• git
• github
• html
• java
• javascript
• linux
• machine learning
• mysql
• numpy
• pandas
• postgresql
• power bi
• prisma
• python
• react
• rest api
• sql
• sql server
• supabase


In [51]:
jd_skills = extract_resume_skills(job_description)

print("JD Skills")
print("="*40)

for skill in jd_skills:
    print("•", skill)

JD Skills
• docker
• fastapi
• git
• linux
• machine learning
• python
• sql


In [52]:
matched_skills = sorted(
    set(resume_skills) &
    set(jd_skills)
)

missing_skills = sorted(
    set(jd_skills) -
    set(resume_skills)
)

In [53]:
if len(jd_skills) > 0:

    skill_score = round(
        len(matched_skills) /
        len(jd_skills) * 100,
        2
    )

else:

    skill_score = 0

In [54]:
print("="*50)

print("Matched Skills\n")

for skill in matched_skills:
    print("✓", skill)

print("\nMissing Skills\n")

for skill in missing_skills:
    print("-", skill)

print("\nSkill Match :", skill_score,"%")

Matched Skills

✓ git
✓ linux
✓ machine learning
✓ python
✓ sql

Missing Skills

- docker
- fastapi

Skill Match : 71.43 %


# Section 10 – ATS Score Calculation

This module combines semantic similarity and technical skill matching to calculate the overall ATS compatibility score. The score provides an estimate of how well the uploaded resume matches the given job description.

In [40]:
def calculate_ats_score(skill_score, semantic_score):
    ats = (0.6 * skill_score) + (0.4 * semantic_score)
    return round(ats, 2)

In [55]:
overall_ats_score = calculate_ats_score(
    skill_score,
    semantic_percentage
)

print("=" * 50)
print(f"Overall ATS Score : {overall_ats_score}%")

Overall ATS Score : 59.42%


In [56]:
print("\nResume Evaluation Summary")
print("="*50)

print("Semantic Match :", semantic_percentage,"%")

print("Skill Match :", skill_score,"%")

print("Overall ATS :", overall_ats_score,"%")


Resume Evaluation Summary
Semantic Match : 41.4 %
Skill Match : 71.43 %
Overall ATS : 59.42 %


# Section 12 – Gemini AI Configuration

This module configures Google's Gemini Large Language Model (LLM), which generates AI-powered resume feedback and powers the resume chatbot.

In [59]:
gemini_model = genai.GenerativeModel("gemini-2.5-flash")

print("Gemini Model Initialized Successfully!")

Gemini Model Initialized Successfully!


In [62]:
response = gemini_model.generate_content(
    "Say hello in one sentence."
)

print(response.text)

Hello!


# Section 13 – Core AI Engine

This section contains the core functions responsible for resume evaluation, including role prediction, semantic matching, skill extraction, ATS score calculation, AI feedback generation, and chatbot interaction.

In [63]:
def predict_role(resume_text):
    """
    Predicts the most suitable job role
    using the trained ML classifier.
    """

    cleaned = clean_text(resume_text)

    vector = vectorizer.transform([cleaned])

    prediction = model.predict(vector)[0]

    probabilities = model.predict_proba(vector)[0]

    predicted_role = label_encoder.inverse_transform(
        [prediction]
    )[0]

    top_indices = probabilities.argsort()[-3:][::-1]

    top_predictions = []

    for idx in top_indices:

        top_predictions.append({

            "role": label_encoder.inverse_transform([idx])[0],

            "confidence": round(
                probabilities[idx] * 100,
                2
            )

        })

    return predicted_role, top_predictions

In [64]:
def calculate_semantic_match(
    resume_text,
    job_description
):
    """
    Calculates semantic similarity
    using Sentence-BERT.
    """

    resume_embedding = semantic_model.encode(
        resume_text,
        convert_to_tensor=True
    )

    jd_embedding = semantic_model.encode(
        job_description,
        convert_to_tensor=True
    )

    similarity = util.cos_sim(
        resume_embedding,
        jd_embedding
    ).item()

    semantic_percentage = round(
        similarity * 100,
        2
    )

    return semantic_percentage

In [65]:
def get_resume_skills(text):
    """
    Extracts technical skills
    from any given text.
    """

    text = text.lower()

    extracted = set()

    for skill in master_skills:

        pattern = r"\b" + re.escape(skill) + r"\b"

        if re.search(pattern, text):
            extracted.add(skill)

    return sorted(extracted)

In [66]:
def get_skill_match(
    resume_text,
    job_description
):
    """
    Compares resume skills
    with JD skills.
    """

    resume_skills = get_resume_skills(
        resume_text
    )

    jd_skills = get_resume_skills(
        job_description
    )

    matched_skills = sorted(
        set(resume_skills) &
        set(jd_skills)
    )

    missing_skills = sorted(
        set(jd_skills) -
        set(resume_skills)
    )

    if len(jd_skills) == 0:

        skill_score = 0

    else:

        skill_score = round(

            len(matched_skills)
            /
            len(jd_skills)
            * 100,

            2

        )

    return (

        matched_skills,

        missing_skills,

        skill_score

    )

In [67]:
def build_rag_context(
    predicted_role,
    top_predictions,
    semantic_match,
    skill_match,
    ats_score,
    matched_skills,
    missing_skills
):
    """
    Builds a structured context
    for Gemini AI.
    """

    top_roles = ", ".join(
        [
            f"{item['role']} ({item['confidence']}%)"
            for item in top_predictions
        ]
    )

    matched = ", ".join(matched_skills)

    missing = ", ".join(missing_skills)

    context = f"""
Resume Evaluation Report

Predicted Role:
{predicted_role}

Top Matching Roles:
{top_roles}

Semantic Match:
{semantic_match}%

Skill Match:
{skill_match}%

Overall ATS Score:
{ats_score}%

Matched Skills:
{matched}

Missing Skills:
{missing}
"""

    return context

In [68]:
import json

def generate_ai_feedback(context_text):
    """
    Generates AI-powered
    resume feedback
    using Gemini.
    """

    prompt = f"""
You are an expert ATS recruiter.

Analyze ONLY the information provided below.

{context_text}

Return ONLY a valid JSON.

Use this format:

{{
    "strengths":[
        "...",
        "...",
        "..."
    ],

    "weaknesses":[
        "...",
        "...",
        "..."
    ],

    "skill_gap_analysis":"...",

    "suggestions":[
        "...",
        "...",
        "..."
    ],

    "hiring_recommendation":{{
        "status":"Excellent Fit | Good Fit | Moderate Fit | Needs Improvement",
        "reason":"..."
    }}
}}

Rules:

- Return ONLY JSON.
- No Markdown.
- No code block.
- No explanation.
- Do not invent skills.
"""

    response = gemini_model.generate_content(prompt)

    try:

        cleaned = response.text.strip()

        cleaned = cleaned.replace(
            "```json",
            ""
        )

        cleaned = cleaned.replace(
            "```",
            ""
        )

        cleaned = cleaned.strip()

        return json.loads(cleaned)

    except Exception as e:

        print("JSON Parsing Error:", e)

        print(response.text)

        return None

In [69]:
def evaluate_resume(resume_text, job_description):
    """
    Complete Resume Evaluation Pipeline
    """

    # ===============================
    # Step 1 : Predict Resume Role
    # ===============================
    predicted_role, top_predictions = predict_role(
        resume_text
    )

    # ===============================
    # Step 2 : Semantic Matching
    # ===============================
    semantic_percentage = calculate_semantic_match(
        resume_text,
        job_description
    )

    # ===============================
    # Step 3 : Skill Matching
    # ===============================
    matched_skills, missing_skills, skill_score = get_skill_match(
        resume_text,
        job_description
    )

    # ===============================
    # Step 4 : ATS Score
    # ===============================
    overall_ats_score = calculate_ats_score(
        skill_score,
        semantic_percentage
    )

    # ===============================
    # Step 5 : Build Context
    # ===============================
    context = build_rag_context(
        predicted_role,
        top_predictions,
        semantic_percentage,
        skill_score,
        overall_ats_score,
        matched_skills,
        missing_skills
    )

    # ===============================
    # Step 6 : Gemini Feedback
    # ===============================
    ai_feedback = generate_ai_feedback(
        context
    )

    # ===============================
    # Step 7 : Return Results
    # ===============================
    return {

        "predicted_role": predicted_role,

        "top_predictions": top_predictions,

        "semantic_match": semantic_percentage,

        "skill_match": skill_score,

        "ats_score": overall_ats_score,

        "matched_skills": matched_skills,

        "missing_skills": missing_skills,

        "ai_feedback": ai_feedback,

        "context": context

    }

In [70]:
def ask_resume_chatbot(question, context):
    """
    Resume AI Chatbot
    """

    prompt = f"""
You are an expert ATS Career Assistant.

Use ONLY the Resume Evaluation Report below
to answer the user's question.

Resume Evaluation Report

{context}

User Question:

{question}

Rules:

- Answer professionally.
- Keep the answer concise.
- If information is unavailable,
  clearly say so.
"""

    response = gemini_model.generate_content(
        prompt
    )

    return response.text

In [71]:
result = evaluate_resume(
    resume_text,
    job_description
)

In [72]:
from pprint import pprint

pprint(result)

{'ai_feedback': {'hiring_recommendation': {'reason': 'The candidate possesses '
                                                     'a solid foundation of '
                                                     'technical skills (71.43% '
                                                     'Skill Match) and several '
                                                     'core development tools '
                                                     '(git, linux, python, '
                                                     'sql). However, the low '
                                                     'Semantic Match (41.4%) '
                                                     'and moderate Overall ATS '
                                                     'Score (59.42%) indicate '
                                                     'the resume does not '
                                                     'strongly align with the '
                                                 

In [73]:
print("=" * 60)
print("Resume Evaluation Report")
print("=" * 60)

print("Predicted Role:")
print(result["predicted_role"])

print("\nTop Matching Roles:")
for item in result["top_predictions"]:
    print(f"- {item['role']} ({item['confidence']}%)")

print("\nSemantic Match:")
print(result["semantic_match"], "%")

print("\nSkill Match:")
print(result["skill_match"], "%")

print("\nOverall ATS Score:")
print(result["ats_score"], "%")

print("\nMatched Skills:")
print(", ".join(result["matched_skills"]))

print("\nMissing Skills:")
print(", ".join(result["missing_skills"]))

Resume Evaluation Report
Predicted Role:
React Developer

Top Matching Roles:
- React Developer (21.52%)
- Python Developer (16.0%)
- Web Designing (9.4%)

Semantic Match:
41.4 %

Skill Match:
71.43 %

Overall ATS Score:
59.42 %

Matched Skills:
git, linux, machine learning, python, sql

Missing Skills:
docker, fastapi


In [74]:
feedback = result["ai_feedback"]

print("=" * 60)
print("AI Resume Feedback")
print("=" * 60)

if feedback:

    print("\nStrengths:")
    for item in feedback["strengths"]:
        print("-", item)

    print("\nWeaknesses:")
    for item in feedback["weaknesses"]:
        print("-", item)

    print("\nSkill Gap Analysis:")
    print(feedback["skill_gap_analysis"])

    print("\nSuggestions:")
    for item in feedback["suggestions"]:
        print("-", item)

    print("\nHiring Recommendation:")
    print(feedback["hiring_recommendation"]["status"])

    print(feedback["hiring_recommendation"]["reason"])

else:

    print("Unable to generate AI feedback.")

AI Resume Feedback

Strengths:
- Strong foundational technical skills demonstrated by matched skills: git, linux, python, sql, machine learning.
- High Skill Match percentage (71.43%) indicates a good number of relevant technical skills are present.
- The resume is predicted for and most closely matches the 'React Developer' role, indicating some core alignment.

Weaknesses:
- Low Semantic Match (41.4%) suggests the experience descriptions and project context do not strongly align with typical React Developer role expectations.
- Moderate Overall ATS Score (59.42%) indicates significant room for improvement in overall keyword optimization and relevance.
- Critical skills for modern development, specifically 'docker' and 'fastapi', are explicitly missing.
- The top matching role percentage for React Developer is relatively low (21.52%), and other roles like Python Developer are also prominent, suggesting a less focused profile for a dedicated React role.

Skill Gap Analysis:
The candida

# Section 15 – AI Resume Chatbot

This module allows users to interact with the AI Resume Evaluation System through natural language questions. The chatbot uses the generated resume evaluation report (RAG context) along with Google's Gemini model to answer resume-related queries, provide career guidance, explain ATS scores, highlight missing skills, and suggest resume improvements.

In [75]:
context = result["context"]

print("Chat Context Ready!")

Chat Context Ready!


In [76]:
print("=" * 60)
print("AI Resume Chatbot")
print("=" * 60)
print("Type 'exit' to end the conversation.\n")

while True:

    question = input("You: ")

    if question.lower() == "exit":
        print("\nSession Ended.")
        break

    answer = ask_resume_chatbot(
        question,
        context
    )

    print("\nAI:", answer)
    print("-" * 60)

AI Resume Chatbot
Type 'exit' to end the conversation.


AI: Based on the Resume Evaluation Report, to improve your resume, it is recommended to incorporate the following missing skills: Docker and FastAPI.
------------------------------------------------------------

AI: Based on the provided Resume Evaluation Report, there is no information regarding specific projects you should add. The report identifies missing skills (docker, fastapi), which could guide your project choices, but does not recommend projects directly.
------------------------------------------------------------
You: exit

Session Ended.
